# Alpha Vantage Exploration for Bronze Layer

This notebook explores the Alpha Vantage datasets that are the most useful for the financial data platform using only endpoints available in the free plan.

Project goal for this notebook:
- explore stock daily prices
- inspect company overview and fundamentals
- inspect FX daily data
- inspect one macroeconomic indicator
- design bronze payloads and file layout for S3


## Step 1. What we want from Alpha Vantage

Alpha Vantage is a strong complement to Yahoo Finance because it gives us more structured datasets for:
- fundamentals
- FX
- macroeconomic indicators
- daily stock prices

Recommended first datasets for this project:
- `TIME_SERIES_DAILY`
- `OVERVIEW`
- `INCOME_STATEMENT`
- `BALANCE_SHEET`
- `FX_DAILY`
- `REAL_GDP`

Important note:
- as of March 23, 2026, `TIME_SERIES_DAILY_ADJUSTED` is documented by Alpha Vantage as a premium endpoint
- for the free plan, we will use `TIME_SERIES_DAILY` here and rely on Yahoo Finance for split and dividend exploration

Official documentation:
- https://www.alphavantage.co/documentation/

## Step 2. Configure the API key

Put your Alpha Vantage key in a `.env` file at the project root.

Example:

```env
ALPHA_VANTAGE_API_KEY=your_key_here
```

If the key is missing, the notebook will raise an error so you notice it early.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv

In [ ]:
load_dotenv()

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
BASE_URL = "https://www.alphavantage.co/query"

if not API_KEY:
    raise ValueError("ALPHA_VANTAGE_API_KEY not found. Add it to your .env file before running this notebook.")

stock_symbol = "IBM"
fx_from = "USD"
fx_to = "BRL"

print("Stock symbol:", stock_symbol)
print("FX pair:", f"{fx_from}/{fx_to}")

## Step 3. Create a reusable request helper

This helper keeps the exploration notebook clean and gives us a structure that later can become a Python client in `src/clients/alpha_vantage.py`.

In [ ]:
def call_alpha_vantage(**params):
    request_params = {**params, "apikey": API_KEY}
    response = requests.get(BASE_URL, params=request_params, timeout=30)
    response.raise_for_status()

    if "application/json" in response.headers.get("Content-Type", ""):
        payload = response.json()
    else:
        payload = response.text

    return {
        "request_url": response.url.replace(API_KEY, "***REDACTED***"),
        "status_code": response.status_code,
        "payload": payload,
    }


def inspect_alpha_payload(payload):
    if isinstance(payload, dict):
        for key in ["Information", "Note", "Error Message"]:
            if key in payload:
                print(f"{key}: {payload[key]}")
    else:
        print("Non-JSON payload received")

## Step 4. Explore free daily stock prices

We will use `TIME_SERIES_DAILY`, which is part of the free documentation.

This dataset gives us:
- open
- high
- low
- close
- volume

This is enough for bronze ingestion and for many silver and gold transformations. For dividend and split data, Yahoo Finance is still the better free source in this project.

In [ ]:
daily_response = call_alpha_vantage(
    function="TIME_SERIES_DAILY",
    symbol=stock_symbol,
    outputsize="compact",
)

print("Status code:", daily_response["status_code"])
print("Request URL:", daily_response["request_url"])
inspect_alpha_payload(daily_response["payload"])

if isinstance(daily_response["payload"], dict):
    print(list(daily_response["payload"].keys())[:10])

In [ ]:
daily_payload = daily_response["payload"]
daily_payload.get("Meta Data", {}) if isinstance(daily_payload, dict) else {}

In [ ]:
time_series_key = "Time Series (Daily)"
daily_series = daily_payload.get(time_series_key, {}) if isinstance(daily_payload, dict) else {}

daily_prices = pd.DataFrame.from_dict(daily_series, orient="index")
daily_prices = daily_prices.reset_index().rename(columns={"index": "price_date"})
daily_prices.columns = [
    column.replace("1. ", "")
    .replace("2. ", "")
    .replace("3. ", "")
    .replace("4. ", "")
    .replace("5. ", "")
    .replace(" ", "_")
    .lower()
    for column in daily_prices.columns
]

if not daily_prices.empty:
    daily_prices["symbol"] = stock_symbol
    daily_prices["price_date"] = pd.to_datetime(daily_prices["price_date"])
    numeric_columns = [column for column in daily_prices.columns if column not in ["symbol", "price_date"]]
    daily_prices[numeric_columns] = daily_prices[numeric_columns].apply(pd.to_numeric, errors="coerce")
    daily_prices = daily_prices.sort_values("price_date", ascending=False)

daily_prices.head()

In [ ]:
if not daily_prices.empty:
    print("Row count:", len(daily_prices))
    print("Date range:", daily_prices["price_date"].min(), "to", daily_prices["price_date"].max())
    print("Duplicate business key count:", daily_prices.duplicated(subset=["symbol", "price_date"]).sum())
    print("Null counts:")
    display(daily_prices.isna().sum())
else:
    print("No daily price rows returned. Check the payload message above.")

## Step 5. Explore company overview

The `OVERVIEW` dataset is useful for attributes that may later become dimensions or slowly changing reference data, such as:
- company name
- sector
- industry
- country
- market capitalization
- valuation ratios
- dividend information

In [ ]:
time.sleep(12)
overview_response = call_alpha_vantage(
    function="OVERVIEW",
    symbol=stock_symbol,
)

overview_payload = overview_response["payload"]
inspect_alpha_payload(overview_payload)

pd.Series({
    "Symbol": overview_payload.get("Symbol") if isinstance(overview_payload, dict) else None,
    "Name": overview_payload.get("Name") if isinstance(overview_payload, dict) else None,
    "Sector": overview_payload.get("Sector") if isinstance(overview_payload, dict) else None,
    "Industry": overview_payload.get("Industry") if isinstance(overview_payload, dict) else None,
    "Country": overview_payload.get("Country") if isinstance(overview_payload, dict) else None,
    "Currency": overview_payload.get("Currency") if isinstance(overview_payload, dict) else None,
    "MarketCapitalization": overview_payload.get("MarketCapitalization") if isinstance(overview_payload, dict) else None,
    "PERatio": overview_payload.get("PERatio") if isinstance(overview_payload, dict) else None,
    "DividendYield": overview_payload.get("DividendYield") if isinstance(overview_payload, dict) else None,
    "52WeekHigh": overview_payload.get("52WeekHigh") if isinstance(overview_payload, dict) else None,
    "52WeekLow": overview_payload.get("52WeekLow") if isinstance(overview_payload, dict) else None,
})

## Step 6. Explore financial statements

The two most useful fundamental datasets to inspect first are:
- `INCOME_STATEMENT`
- `BALANCE_SHEET`

They are helpful because they support future gold datasets such as profitability, leverage, and trend metrics.

In [ ]:
time.sleep(12)
income_statement_response = call_alpha_vantage(
    function="INCOME_STATEMENT",
    symbol=stock_symbol,
)

income_statement_payload = income_statement_response["payload"]
inspect_alpha_payload(income_statement_payload)
list(income_statement_payload.keys()) if isinstance(income_statement_payload, dict) else []

In [ ]:
quarterly_income = pd.DataFrame(income_statement_payload.get("quarterlyReports", [])) if isinstance(income_statement_payload, dict) else pd.DataFrame()
quarterly_income.head()

In [ ]:
time.sleep(12)
balance_sheet_response = call_alpha_vantage(
    function="BALANCE_SHEET",
    symbol=stock_symbol,
)

balance_sheet_payload = balance_sheet_response["payload"]
inspect_alpha_payload(balance_sheet_payload)
quarterly_balance = pd.DataFrame(balance_sheet_payload.get("quarterlyReports", [])) if isinstance(balance_sheet_payload, dict) else pd.DataFrame()
quarterly_balance.head()

## Step 7. Explore FX daily rates

FX adds a global market angle to your platform and is very useful for BI. A simple pair like `USD/BRL` already improves the story of the project.

In [ ]:
time.sleep(12)
fx_daily_response = call_alpha_vantage(
    function="FX_DAILY",
    from_symbol=fx_from,
    to_symbol=fx_to,
    outputsize="compact",
)

fx_payload = fx_daily_response["payload"]
inspect_alpha_payload(fx_payload)
list(fx_payload.keys()) if isinstance(fx_payload, dict) else []

In [ ]:
fx_series = fx_payload.get("Time Series FX (Daily)", {}) if isinstance(fx_payload, dict) else {}
fx_daily = pd.DataFrame.from_dict(fx_series, orient="index")
fx_daily = fx_daily.reset_index().rename(columns={"index": "fx_date"})
fx_daily.columns = [
    column.replace("1. ", "")
    .replace("2. ", "")
    .replace("3. ", "")
    .replace("4. ", "")
    .replace(" ", "_")
    .lower()
    for column in fx_daily.columns
]

if not fx_daily.empty:
    fx_daily["from_symbol"] = fx_from
    fx_daily["to_symbol"] = fx_to
    fx_daily["fx_date"] = pd.to_datetime(fx_daily["fx_date"])
    numeric_columns = [column for column in fx_daily.columns if column not in ["fx_date", "from_symbol", "to_symbol"]]
    fx_daily[numeric_columns] = fx_daily[numeric_columns].apply(pd.to_numeric, errors="coerce")
    fx_daily = fx_daily.sort_values("fx_date", ascending=False)

fx_daily.head()

## Step 8. Explore one macroeconomic indicator

Macro indicators are valuable because they help you tell a stronger analytical story in the gold layer. We will start with `REAL_GDP`.

In [ ]:
time.sleep(12)
real_gdp_response = call_alpha_vantage(function="REAL_GDP")
real_gdp_payload = real_gdp_response["payload"]
inspect_alpha_payload(real_gdp_payload)
list(real_gdp_payload.keys()) if isinstance(real_gdp_payload, dict) else []

In [ ]:
real_gdp = pd.DataFrame(real_gdp_payload.get("data", [])) if isinstance(real_gdp_payload, dict) else pd.DataFrame()

if not real_gdp.empty:
    real_gdp["date"] = pd.to_datetime(real_gdp["date"])
    real_gdp["value"] = pd.to_numeric(real_gdp["value"], errors="coerce")
    real_gdp = real_gdp.sort_values("date", ascending=False)

real_gdp.head()

## Step 9. Free-plan notes

When using the free plan, keep these exploration habits:
- do not fire many cells quickly one after another
- wait between calls when needed
- always inspect `Information`, `Note`, and `Error Message` fields
- treat those messages as first-class bronze metadata in your ingestion logs

This is why the notebook includes `time.sleep(12)` before later requests.

## Step 10. Bronze layer design

Recommended bronze datasets from Alpha Vantage using the free plan:
- `stock_prices_daily`
- `company_overview`
- `income_statement`
- `balance_sheet`
- `fx_daily`
- `macro_real_gdp`

Recommended raw path layout:

`s3://financial-data/bronze/source=alpha_vantage/dataset=stock_prices_daily/symbol=IBM/ingestion_date=2026-03-23/file.json`

Important metadata to store with each bronze record:
- source
- dataset
- natural key such as symbol or currency pair
- ingestion timestamp
- request parameters
- request URL
- raw payload


In [ ]:
ingestion_ts = datetime.now(timezone.utc).isoformat()

bronze_stock_prices_payload = {
    "source": "alpha_vantage",
    "dataset": "stock_prices_daily",
    "symbol": stock_symbol,
    "ingestion_ts_utc": ingestion_ts,
    "request_params": {
        "function": "TIME_SERIES_DAILY",
        "symbol": stock_symbol,
        "outputsize": "compact",
    },
    "request_url": daily_response["request_url"],
    "raw_payload": daily_payload,
}

print(json.dumps(bronze_stock_prices_payload, indent=2, default=str)[:2500])

In [ ]:
ingestion_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

stock_prices_path = Path(
    f"bronze/source=alpha_vantage/dataset=stock_prices_daily/symbol={stock_symbol}/ingestion_date={ingestion_date}/daily.json"
)
overview_path = Path(
    f"bronze/source=alpha_vantage/dataset=company_overview/symbol={stock_symbol}/ingestion_date={ingestion_date}/overview.json"
)
fx_path = Path(
    f"bronze/source=alpha_vantage/dataset=fx_daily/from_symbol={fx_from}/to_symbol={fx_to}/ingestion_date={ingestion_date}/fx_daily.json"
)

print(stock_prices_path.as_posix())
print(overview_path.as_posix())
print(fx_path.as_posix())

## Step 11. What we learned from this notebook

Best free Alpha Vantage datasets for this project:
- daily stock prices for market facts
- company overview for descriptive reference data
- financial statements for future analytical metrics
- FX for global market coverage
- macro indicators for richer gold-layer analysis

Best use of Alpha Vantage in your architecture:
- bronze: land raw JSON payloads exactly as returned
- silver: normalize price, FX, and fundamentals into typed tabular datasets
- gold: combine with Yahoo Finance market coverage and analytical models

Important tradeoff:
- Yahoo Finance is your better free source for dividends and splits
- Alpha Vantage free endpoints are still excellent for fundamentals, FX, and macro data

Suggested next build step after this notebook:
- create a small Python ingestion module that saves these raw payloads locally using the same bronze partition pattern we plan to use in S3